# Notebook 07 — STE Variants Evaluation

Four variants of the Straight-Through Estimator optimizer, motivated by lessons from Proximal and LM_hybrid:

| Variant | Key change | Motivation |
|---|---|---|
| **STE_base** | Original: Adam + cosine LR, MSE-only, clamp ±1.5 | Baseline |
| **STE_reg** | + ternary reg w²(1-w²) with λ warm-up, clamp ±1 | Push latent weights away from snap threshold |
| **STE_dual** | STE_reg + dual stopping (mse AND boundary_frac) | Stop only when ternary solution is stable |
| **STE_hybrid** | Phase 1 (pure MSE) → Phase 2 (reg + dual stop) | Find good solution first, then stabilize |

**Key metric — `boundary_frac`**: fraction of latent weights within ±0.15 of the snap threshold (±0.33).  
These weights are unstable under crystallization — a small perturbation flips which ternary bin they snap to.  
Lower boundary_frac → more stable crystallization.

Evaluated on: **MONK-1** (17 features, 124 train) and **Mushroom** (111 features, 8124 train).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from luknn.layers.lukasiewicz_linear import make_lukasiewicz_net
from luknn.benchmark.datasets import load_monk, load_mushroom
from luknn.benchmark.metrics import compute_f1, compute_accuracy, compute_delta_n
from luknn.optimizers import (
    STEOptimizer, STERegOptimizer, STEDualOptimizer, STEHybridOptimizer,
)
from luknn.training.ste import boundary_frac

print('Setup OK')

## 1 · Parameters

In [ ]:
BASE_SEED = 42
N_TRIALS  = 10
MAX_ITER  = 2000
TOL_MSE   = 2e-3

RESULTS_DIR = '../results/ste_variants'
os.makedirs(RESULTS_DIR, exist_ok=True)

def _make_model(n_features):
    return make_lukasiewicz_net(
        n_features, n_hidden_layers=2, hidden_width=n_features, mode='ste'
    )

# Dual stop: mse < MSE_GATE AND boundary_frac < TOL_BOUNDARY
# MSE_GATE=0.05 is realistic for STE (quantized forward pass never hits tol_mse=2e-3).
# TOL_BOUNDARY=0.35 matches the bf_pre plateau observed for STE_reg (~0.306).
MSE_GATE      = 0.05
TOL_BOUNDARY  = 0.35

VARIANT_DEFS = {
    'STE_base':   lambda m: STEOptimizer(m, lr=5e-3),
    'STE_reg':    lambda m: STERegOptimizer(m, lr=5e-3, lambda_attract=0.05),
    'STE_dual':   lambda m: STEDualOptimizer(m, lr=5e-3, lambda_attract=0.05,
                                             mse_gate=MSE_GATE, tol_boundary=TOL_BOUNDARY),
    'STE_hybrid': lambda m: STEHybridOptimizer(m, lr=5e-3, lambda_attract=0.05,
                                               mse_gate=MSE_GATE, tol_boundary=TOL_BOUNDARY,
                                               p1_fraction=0.4),
}

VARIANT_COLORS = {
    'STE_base':   'steelblue',
    'STE_reg':    'darkorange',
    'STE_dual':   'seagreen',
    'STE_hybrid': 'purple',
}

print('Parameters set.')

## 2 · Helpers

In [ ]:
def run_trial(variant_name, make_opt, ds, seed, max_iter, tol_mse):
    torch.manual_seed(seed)
    model = _make_model(ds.n_features)
    opt   = make_opt(model)

    res = opt.train(ds.X_train, ds.y_train, tol_mse=tol_mse, max_iter=max_iter)

    # After train(), model is already crystallized by the optimizer.
    with torch.no_grad():
        pred = model(ds.X_test)

    f1  = compute_f1(pred, ds.y_test)
    acc = compute_accuracy(pred, ds.y_test)
    dn  = compute_delta_n(model)

    return {
        'variant':    variant_name,
        'seed':       seed,
        'f1':         round(f1, 4),
        'acc':        round(acc, 4),
        'dn_post':    round(dn, 4),
        'bf_pre':     round(res.extra.get('boundary_frac_pre', 0.0), 4),
        'converged':  res.converged,
        'iterations': res.iterations,
        'final_mse':  round(res.final_mse, 6),
        'time_s':     round(res.total_time_s, 2),
        'reason':     res.reason,
        'mse_history': res.mse_history,
    }

## 3 · MONK-1 benchmark

In [ ]:
MONK_CSV = f'{RESULTS_DIR}/monk_ste_variants.csv'

if os.path.exists(MONK_CSV):
    monk_df = pd.read_csv(MONK_CSV)
    print(f'Loaded pre-computed MONK results from {MONK_CSV}')
else:
    ds_monk = load_monk(problem=1, seed=BASE_SEED)
    print(f'MONK-1: features={ds_monk.n_features}  train={len(ds_monk.X_train)}  test={len(ds_monk.X_test)}')

    monk_rows = []
    for vname, make_opt in VARIANT_DEFS.items():
        print(f'\n[{vname}]')
        for trial in range(N_TRIALS):
            seed = BASE_SEED + trial * 17
            r = run_trial(vname, make_opt, ds_monk, seed, MAX_ITER, TOL_MSE)
            row = {k: v for k, v in r.items() if k != 'mse_history'}
            monk_rows.append(row)
            print(f'  trial {trial:2d}: F1={r["f1"]:.3f}  bf_pre={r["bf_pre"]:.3f}  '
                  f'iters={r["iterations"]:4d}  t={r["time_s"]:.1f}s  {r["reason"]}')

    monk_df = pd.DataFrame(monk_rows)
    monk_df.to_csv(MONK_CSV, index=False)
    print('\nDone.')

In [ ]:
agg = monk_df.groupby('variant').agg(
    f1_mean     = ('f1',       'mean'),
    f1_std      = ('f1',       'std'),
    bf_mean     = ('bf_pre',   'mean'),
    bf_std      = ('bf_pre',   'std'),
    conv_rate   = ('converged','mean'),
    iters_mean  = ('iterations','mean'),
    time_mean   = ('time_s',   'mean'),
).round(3)
agg = agg.reindex(list(VARIANT_DEFS.keys()))
print('MONK-1 Summary')
agg

In [ ]:
order  = list(VARIANT_DEFS.keys())
colors = [VARIANT_COLORS[v] for v in order]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('MONK-1 — STE Variants Comparison', fontsize=13)

for ax, col, title in zip(axes,
    ['f1', 'bf_pre', 'dn_post'],
    ['F1 (post-crystallization)', 'boundary_frac (pre-crystallization)', 'Δ(N) post-crystallization']):
    data = [monk_df[monk_df['variant'] == v][col].values for v in order]
    bp = ax.boxplot(data, patch_artist=True, labels=order)
    for patch, col_ in zip(bp['boxes'], colors):
        patch.set_facecolor(col_)
        patch.set_alpha(0.6)
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=12)
    ax.grid(True, alpha=0.3)
    if col == 'f1':
        ax.set_ylim(0, 1.05)
        ax.set_ylabel('F1')
    elif col == 'bf_pre':
        ax.set_ylabel('boundary_frac')
    else:
        ax.set_ylabel('Δ(N)')

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/monk_ste_boxplot.png', dpi=150)
plt.show()

In [ ]:
ds_monk = load_monk(problem=1, seed=BASE_SEED)

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle('MONK-1 — MSE learning curves (seed=42)', fontsize=12)
axes = axes.flatten()

for ax, (vname, make_opt) in zip(axes, VARIANT_DEFS.items()):
    torch.manual_seed(BASE_SEED)
    model = _make_model(ds_monk.n_features)
    opt   = make_opt(model)
    res   = opt.train(ds_monk.X_train, ds_monk.y_train, tol_mse=TOL_MSE, max_iter=MAX_ITER)
    ax.semilogy(res.mse_history, color=VARIANT_COLORS[vname], lw=1.5)
    ax.axhline(TOL_MSE, color='grey', ls='--', lw=1, label=f'tol={TOL_MSE}')
    ax.set_title(f'{vname}  (reason={res.reason})')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('MSE (log)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/monk_ste_curves.png', dpi=150)
plt.show()

## 4 · Mushroom benchmark (111 features)

In [ ]:
MUSH_CSV = f'{RESULTS_DIR}/mushroom_ste_variants.csv'

if os.path.exists(MUSH_CSV):
    mush_df = pd.read_csv(MUSH_CSV)
    print(f'Loaded pre-computed Mushroom results from {MUSH_CSV}')
else:
    ds_mush = load_mushroom(seed=BASE_SEED)
    print(f'Mushroom: features={ds_mush.n_features}  train={len(ds_mush.X_train)}  '
          f'test={len(ds_mush.X_test)}')

    mush_rows = []
    for vname, make_opt in VARIANT_DEFS.items():
        print(f'\n[{vname}]')
        for trial in range(5):
            seed = BASE_SEED + trial * 17
            r = run_trial(vname, make_opt, ds_mush, seed, max_iter=1000, tol_mse=TOL_MSE)
            row = {k: v for k, v in r.items() if k != 'mse_history'}
            mush_rows.append(row)
            print(f'  trial {trial:2d}: F1={r["f1"]:.3f}  bf_pre={r["bf_pre"]:.3f}  '
                  f'iters={r["iterations"]:4d}  t={r["time_s"]:.1f}s  {r["reason"]}')

    mush_df = pd.DataFrame(mush_rows)
    mush_df.to_csv(MUSH_CSV, index=False)
    print('\nDone.')

In [ ]:
agg_mush = mush_df.groupby('variant').agg(
    f1_mean     = ('f1',       'mean'),
    f1_std      = ('f1',       'std'),
    bf_mean     = ('bf_pre',   'mean'),
    bf_std      = ('bf_pre',   'std'),
    conv_rate   = ('converged','mean'),
    iters_mean  = ('iterations','mean'),
    time_mean   = ('time_s',   'mean'),
).round(3)
agg_mush = agg_mush.reindex(list(VARIANT_DEFS.keys()))
print('Mushroom Summary')
agg_mush

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Mushroom — STE Variants Comparison', fontsize=13)

for ax, col, title in zip(axes,
    ['f1', 'bf_pre', 'dn_post'],
    ['F1 (post-crystallization)', 'boundary_frac (pre-crystallization)', 'Δ(N) post']):
    data = [mush_df[mush_df['variant'] == v][col].values for v in order]
    bp = ax.boxplot(data, patch_artist=True, labels=order)
    for patch, col_ in zip(bp['boxes'], colors):
        patch.set_facecolor(col_)
        patch.set_alpha(0.6)
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=12)
    ax.grid(True, alpha=0.3)
    if col == 'f1':
        ax.set_ylim(0, 1.05)
        ax.set_ylabel('F1')
    elif col == 'bf_pre':
        ax.set_ylabel('boundary_frac')
    else:
        ax.set_ylabel('Δ(N)')

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/mushroom_ste_boxplot.png', dpi=150)
plt.show()

## 5 · Joint comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (ds_label, df_) in zip(axes, [('MONK-1', monk_df), ('Mushroom', mush_df)]):
    x = np.arange(len(order))
    means = [df_[df_['variant'] == v]['f1'].mean() for v in order]
    stds  = [df_[df_['variant'] == v]['f1'].std()  for v in order]
    bars  = ax.bar(x, means, yerr=stds, capsize=5,
                   color=[VARIANT_COLORS[v] for v in order], alpha=0.75)
    ax.set_xticks(x)
    ax.set_xticklabels(order, rotation=12, ha='right')
    ax.set_ylim(0, 1.15)
    ax.set_ylabel('F1 (post-crystallization)')
    ax.set_title(f'{ds_label} — F1 (mean ± std)')
    ax.grid(axis='y', alpha=0.3)
    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width()/2, mean + 0.02,
                f'{mean:.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/joint_f1.png', dpi=150)
plt.show()

## 6 · boundary_frac analysis

Does lower `boundary_frac` correlate with higher F1?  
If yes, the ternary reg is doing its job of stabilizing the latent weights.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('boundary_frac vs F1 — latent weight stability', fontsize=12)

for ax, (ds_label, df_) in zip(axes, [('MONK-1', monk_df), ('Mushroom', mush_df)]):
    for vname in order:
        g = df_[df_['variant'] == vname]
        ax.scatter(g['bf_pre'], g['f1'],
                   color=VARIANT_COLORS[vname], label=vname, alpha=0.7, s=40)
    ax.set_xlabel('boundary_frac (pre-crystallization)')
    ax.set_ylabel('F1 (post-crystallization)')
    ax.set_title(ds_label)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    corr = df_[['bf_pre', 'f1']].corr().iloc[0, 1]
    ax.text(0.05, 0.05, f'r = {corr:.2f}', transform=ax.transAxes, fontsize=10)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/boundary_vs_f1.png', dpi=150)
plt.show()

## 7 · Conclusions

### MONK-1 (17 features)

| Variant | F1±std | bf_pre | Winner? |
|---|---|---|---|
| STE_base | 0.611±0.333 | 0.442 | — |
| **STE_reg** | **0.734±0.115** | 0.306 | **MONK** |
| STE_dual | 0.704±0.098 | 0.316 | — |
| STE_hybrid | 0.617±0.250 | 0.217 | — |

**STE_reg** is the best on MONK: +12% F1 vs base, lowest variance.  
The ternary reg w²(1-w²) pushes latent weights away from snap thresholds (bf: 0.442→0.306) without destabilising training.  
STE_dual adds marginal stability (lower std) at the cost of 3 F1 points — dual stop occasionally fires before convergence.  
STE_hybrid underperforms: Phase 2 reg disrupts the Phase 1 solution on this low-dimensional task.

### Mushroom (111 features)

| Variant | F1±std | bf_pre | Winner? |
|---|---|---|---|
| STE_base | 0.033±0.035 | 0.428 | — |
| STE_reg | 0.019±0.014 | 0.430 | — |
| STE_dual | 0.019±0.014 | 0.430 | — |
| **STE_hybrid** | **0.516±0.315** | 0.015 | **Mushroom** |

Base/reg/dual all collapse at ~56 iters — the quantized gradient through 8124×111 weights is too noisy for the loss to descend.  
**STE_hybrid** is the only variant that learns (F1 up to 0.803 in best trial).  
Phase 1 (pure MSE, no reg) gives the network 400 iterations to find a solution; Phase 2 then crystallizes it (bf: 0.428→0.015).  
High variance (±0.315) and one complete collapse (trial 4, F1=0.000) indicate instability at the Phase 1→2 transition.

### Key insight — dimension-dependent optimiser choice

The two winners are **complementary**:
- Low-dimensional (MONK): always-on ternary reg (STE_reg) works — the gradient signal is clean enough.  
- High-dimensional (Mushroom): two-phase separation (STE_hybrid) is necessary — learn first, then crystallize.

This mirrors the LM_hybrid finding: a clean continuous solution is a prerequisite for effective ternary regularization.

In [ ]:
print('=== MONK-1 ===')
for v in order:
    g = monk_df[monk_df['variant'] == v]
    print(f'  {v:<14}  F1={g["f1"].mean():.3f}±{g["f1"].std():.3f}  '
          f'bf={g["bf_pre"].mean():.3f}  conv={g["converged"].mean():.0%}')

print()
print('=== Mushroom ===')
for v in order:
    g = mush_df[mush_df['variant'] == v]
    print(f'  {v:<14}  F1={g["f1"].mean():.3f}±{g["f1"].std():.3f}  '
          f'bf={g["bf_pre"].mean():.3f}  conv={g["converged"].mean():.0%}')